In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import copy
import pickle
from tqdm import tqdm # Import tqdm for progress bars

# --- Configuration ---
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LEARNING_RATE = 0.01
GLOBAL_LRS_TO_TEST = [1.0, 1.5, 2.0] # Server-side learning rates for FedGH
BATCH_SIZE = 64
K_FIXED = 5 # Fixed number of local epochs
ALPHA_CHALLENGING = 0.1 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

print(f"Running on device: {DEVICE}")

Running on device: cuda


### Step 1: Model and Data Preparation (Unchanged)

In [2]:
# --- Model Definition for CIFAR-10 ---
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super(CIFAR_CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(8 * 8 * 32, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 8 * 8 * 32)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# --- Data Partitioning Function ---
def dirichlet_partition_data(dataset, num_clients, alpha, batch_size):
    num_classes = len(dataset.classes)
    class_indices = [np.where(np.array(dataset.targets) == i)[0] for i in range(num_classes)]
    client_indices = [[] for _ in range(num_clients)]
    for c_indices in class_indices:
        proportions = np.random.dirichlet([alpha] * num_clients)
        num_samples_for_class = (proportions * len(c_indices)).astype(int)
        rem = len(c_indices) - num_samples_for_class.sum()
        for i in range(rem):
            num_samples_for_class[i % num_clients] += 1
        start = 0
        for i in range(num_clients):
            end = start + num_samples_for_class[i]
            client_indices[i].extend(c_indices[start:end])
            start = end
    client_loaders = []
    for indices in client_indices:
        client_dataset = Subset(dataset, indices)
        loader = DataLoader(client_dataset, batch_size=batch_size, shuffle=True)
        client_loaders.append(loader)
    return client_loaders

# --- Data Loading and Transformation ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
full_train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

### Step 2: Client Training, Evaluation, and FedGH Harmonization Logic

In [3]:
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return 100 * correct / total

def train_client_and_get_delta(model, data_loader, lr, local_epochs):
    initial_state = copy.deepcopy(model.state_dict())
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(local_epochs):
        for data, target in data_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
    
    final_state = model.state_dict()
    model_delta = {name: final_state[name] - initial_state[name] for name in final_state}
    return model_delta

def harmonize_updates(model_deltas):
    num_clients = len(model_deltas)
    if num_clients <= 1:
        return model_deltas

    flat_deltas = []
    shapes = []
    param_keys = list(model_deltas[0].keys())

    for delta_dict in model_deltas:
        client_flat = []
        current_shapes = []
        for key in param_keys:
            param = delta_dict[key]
            client_flat.append(param.view(-1))
            current_shapes.append(param.shape)
        flat_deltas.append(torch.cat(client_flat))
        shapes.append(current_shapes)
    
    # Use original values for projection calculations
    original_flat_deltas = [d.clone() for d in flat_deltas]
    
    for i in range(num_clients):
        for j in range(i + 1, num_clients):
            g_i_orig = original_flat_deltas[i]
            g_j_orig = original_flat_deltas[j]

            dot_product = torch.dot(g_i_orig, g_j_orig)

            if dot_product < 0:
                norm_i_sq = torch.dot(g_i_orig, g_i_orig) + 1e-8
                norm_j_sq = torch.dot(g_j_orig, g_j_orig) + 1e-8

                proj_on_j = (dot_product / norm_j_sq) * g_j_orig
                proj_on_i = (dot_product / norm_i_sq) * g_i_orig

                flat_deltas[i] -= proj_on_j
                flat_deltas[j] -= proj_on_i

    harmonized_deltas = []
    for i, flat_delta in enumerate(flat_deltas):
        new_delta_dict = {}
        start = 0
        client_shapes = shapes[i]
        for key, shape in zip(param_keys, client_shapes):
            num_elements = torch.prod(torch.tensor(shape)).item()
            param = flat_delta[start : start + num_elements].view(shape)
            new_delta_dict[key] = param
            start += num_elements
        harmonized_deltas.append(new_delta_dict)
        
    return harmonized_deltas

### Step 3: Self-Contained Experiment Runner Functions

In [4]:
def run_fedavg_experiment(alpha, num_rounds, num_clients, local_epochs, batch_size, lr, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    result_filename = f'task4_3/results/fedavg_alpha_{alpha}.pkl'
    os.makedirs(os.path.dirname(result_filename), exist_ok=True)

    if os.path.exists(result_filename):
        print(f"\n--- Loading existing FedAvg results from {result_filename} ---")
        with open(result_filename, 'rb') as f:
            return pickle.load(f)['accuracies']

    print(f"\n--- Running FedAvg Experiment (alpha={alpha}) ---")
    client_loaders = dirichlet_partition_data(full_train_dataset, num_clients, alpha, batch_size)
    global_model = CIFAR_CNN().to(DEVICE)
    accuracies = []

    for round_num in range(num_rounds):
        start_time = time.time()
        model_deltas = []
        client_iterator = tqdm(range(num_clients), desc=f"FedAvg Round {round_num+1}/{num_rounds}", leave=False)
        for client_idx in client_iterator:
            local_model = CIFAR_CNN().to(DEVICE)
            local_model.load_state_dict(global_model.state_dict())
            delta = train_client_and_get_delta(local_model, client_loaders[client_idx], lr, local_epochs)
            model_deltas.append(delta)
        
        global_state_dict = global_model.state_dict()
        for key in global_state_dict.keys():
            avg_delta = torch.stack([d[key] for d in model_deltas]).mean(dim=0)
            global_state_dict[key] += avg_delta
        global_model.load_state_dict(global_state_dict)
        
        acc = evaluate_model(global_model, test_loader)
        accuracies.append(acc)
        round_time = time.time() - start_time
        print(f"FedAvg Round {round_num+1}/{num_rounds} | Accuracy: {acc:.2f}% | Time: {round_time:.2f}s")
    
    print(f"--- Saving FedAvg results to {result_filename} ---")
    with open(result_filename, 'wb') as f:
        pickle.dump({'accuracies': accuracies}, f)
    
    return accuracies

def run_fedgh_experiment(alpha, num_rounds, num_clients, local_epochs, batch_size, lr, global_lr, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    result_filename = f'task4_3/results/fedgh_alpha_{alpha}_glr_{global_lr}.pkl'
    os.makedirs(os.path.dirname(result_filename), exist_ok=True)

    if os.path.exists(result_filename):
        print(f"\n--- Loading existing FedGH (glr={global_lr}) results from {result_filename} ---")
        with open(result_filename, 'rb') as f:
            return pickle.load(f)['accuracies']

    print(f"\n--- Running FedGH Experiment (alpha={alpha}, global_lr={global_lr}) ---")
    client_loaders = dirichlet_partition_data(full_train_dataset, num_clients, alpha, batch_size)
    global_model = CIFAR_CNN().to(DEVICE)
    accuracies = []

    for round_num in range(num_rounds):
        start_time = time.time()
        model_deltas = []
        client_iterator = tqdm(range(num_clients), desc=f"FedGH (glr={global_lr}) Round {round_num+1}/{num_rounds}", leave=False)
        for client_idx in client_iterator:
            local_model = CIFAR_CNN().to(DEVICE)
            local_model.load_state_dict(global_model.state_dict())
            delta = train_client_and_get_delta(local_model, client_loaders[client_idx], lr, local_epochs)
            model_deltas.append(delta)

        harmonized_deltas = harmonize_updates(model_deltas)

        global_state_dict = global_model.state_dict()
        for key in global_state_dict.keys():
            avg_delta = torch.stack([d[key] for d in harmonized_deltas]).mean(dim=0)
            global_state_dict[key] += global_lr * avg_delta # Apply global learning rate
        global_model.load_state_dict(global_state_dict)
        
        acc = evaluate_model(global_model, test_loader)
        accuracies.append(acc)
        round_time = time.time() - start_time
        print(f"FedGH Round {round_num+1}/{num_rounds} | Accuracy: {acc:.2f}% | Time: {round_time:.2f}s")

    print(f"--- Saving FedGH (glr={global_lr}) results to {result_filename} ---")
    with open(result_filename, 'wb') as f:
        pickle.dump({'accuracies': accuracies}, f)
        
    return accuracies

### Step 4: Plotting and Main Execution

In [ ]:
def plot_comparison(fedavg_accuracies, fedgh_results, alpha):
    figure_filename = f'task4_3/figures/fedavg_vs_fedgh_alpha_{alpha}.png'
    os.makedirs(os.path.dirname(figure_filename), exist_ok=True)

    plt.figure(figsize=(12, 8))
    plt.style.use('seaborn-v0_8-whitegrid')

    rounds = range(1, len(fedavg_accuracies) + 1)
    plt.plot(rounds, fedavg_accuracies, marker='x', linestyle='--', label=f'FedAvg (Baseline)', zorder=10)
    
    for global_lr, accuracies in fedgh_results.items():
        plt.plot(rounds, accuracies, marker='o', linestyle='-', markersize=4, label=f'FedGH (global_lr={global_lr})')

    plt.title(f'FedAvg vs. FedGH on CIFAR-10 (α={alpha})', fontsize=16)
    plt.xlabel("Communication Round", fontsize=12)
    plt.ylabel("Global Test Accuracy (%)", fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.xticks(np.arange(0, len(rounds) + 1, 5))
    plt.yticks(np.arange(0, 101, 10))
    plt.tight_layout()
    plt.savefig(figure_filename)
    print(f"\nPlot saved to {figure_filename}")
    plt.show()

# --- Main Execution Block ---
if __name__ == '__main__':
    # fedavg_accuracies = run_fedavg_experiment(
    #     alpha=ALPHA_CHALLENGING, num_rounds=NUM_ROUNDS, num_clients=NUM_CLIENTS,
    #     local_epochs=K_FIXED, batch_size=BATCH_SIZE, lr=LEARNING_RATE, seed=SEED)
    
    fedgh_results = {}
    for glr in GLOBAL_LRS_TO_TEST:
        accuracies = run_fedgh_experiment(
            alpha=ALPHA_CHALLENGING, num_rounds=NUM_ROUNDS, num_clients=NUM_CLIENTS,
            local_epochs=K_FIXED, batch_size=BATCH_SIZE, lr=LEARNING_RATE, 
            global_lr=glr, seed=SEED)
        fedgh_results[glr] = accuracies

    # plot_comparison(fedavg_accuracies, fedgh_results, ALPHA_CHALLENGING)


--- Running FedGH Experiment (alpha=0.1, global_lr=1.0) ---


FedGH Round 1/50 | Accuracy: 16.59% | Time: 43.45s


FedGH Round 2/50 | Accuracy: 25.91% | Time: 27.73s


FedGH Round 3/50 | Accuracy: 26.10% | Time: 28.48s


FedGH Round 4/50 | Accuracy: 29.38% | Time: 29.15s


...

FedGH Round 6/50 | Accuracy: 38.15% | Time: 27.72s


FedGH Round 7/50 | Accuracy: 39.51% | Time: 26.55s


FedGH Round 8/50 | Accuracy: 44.05% | Time: 28.83s


FedGH Round 9/50 | Accuracy: 40.04% | Time: 33.41s


FedGH (glr=1.5) Round 10/50:  70%|███████   | 7/10 [00:18<00:10,  3.63s/it]